# Car Semantic Search - Sentence Transformer

## 1. Install dependencies

In [13]:
%pip install -q sentence-transformers pandas kagglehub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import libraries and load the model
Load the model once when starting the notebook. The model remains available for the rest of the session.

In [14]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
print("Loaded model:", MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3440.86it/s]


Loaded model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 3. Download the car dataset from Kaggle

This notebook downloads the public [qucvinhdng/automobile-dataset](https://www.kaggle.com/datasets/qucvinhdng/automobile-dataset) from Kaggle to the KaggleHub cache. You do not need to prepare a local CSV file. The dataset contains the file `data.csv`.

In [15]:
DATASET_HANDLE = "qucvinhdng/automobile-dataset"
dataset_dir = Path(kagglehub.dataset_download(DATASET_HANDLE))
csv_path = dataset_dir / "data.csv"

if not csv_path.is_file():
    raise FileNotFoundError(f"Could not find data.csv in the Kaggle dataset: {dataset_dir}")

raw_df = pd.read_csv(csv_path)
required_columns = {"Name", "Description"}
missing_columns = required_columns.difference(raw_df.columns)
if missing_columns:
    raise ValueError(f"The dataset is missing required columns: {sorted(missing_columns)}")

cars = raw_df.rename(columns={"Name": "name", "Description": "description"}).copy()
cars["name"] = cars["name"].fillna("").astype(str).str.strip()
cars["description"] = cars["description"].fillna("").astype(str).str.strip()
cars = cars.loc[cars["name"].ne("")].reset_index(drop=True)
if cars.empty:
    raise ValueError("The Kaggle dataset contains no valid car listings.")

search_columns = [column for column in cars.columns if column not in {"Link", "Car_code"}]
cars["search_text"] = cars[search_columns].fillna("").astype(str).agg(" | ".join, axis=1)
display(cars[["name", "description"]].head())

,name,description
0,Mercedes Benz C class C200 Exclusive - 2021,"*Xe lắp ráp trong nước, màu xanh, máy xăng 2.0..."
1,Nissan Terra E 2.5 AT 2WD - 2019,"*Xe nhập khẩu, màu vàng, máy xăng 2.5 L, số tự..."
2,Mercedes Benz GLC 300 4Matic - 2021,"*Xe lắp ráp trong nước, màu đen, máy xăng 2.0 ..."
3,Hyundai SantaFe 2.4L 4WD - 2016,"*Xe lắp ráp trong nước, màu bạc, máy xăng 2.4 ..."
4,Kia Carnival Luxury 2.2D - 2022,"*Xe lắp ráp trong nước, màu xanh, máy dầu 2.2 ..."


## 4. Encode car information as vectors

Each car listing is encoded as a fixed-length vector using its description and other available information. The first 8 values are shown for inspection.

In [16]:
car_texts = cars["search_text"].tolist()
car_embeddings = model.encode(car_texts, convert_to_numpy=True, normalize_embeddings=True)
print("Embedding matrix shape:", car_embeddings.shape)
print("First 8 values of the first car vector:", np.round(car_embeddings[0][:8], 4))

Embedding matrix shape: (15442, 384)
First 8 values of the first car vector: [ 0.0268  0.083   0.0127 -0.051  -0.0281  0.0535 -0.0061  0.0593]


## 5. Search for cars using cosine similarity

Because all vectors are normalized to unit length, their dot product equals cosine similarity. Higher scores indicate greater semantic similarity; they are not probabilities.

In [17]:
query = "Tôi muốn tìm xe SUV 7 chỗ rộng rãi cho gia đình đi du lịch"
query_embedding = model.encode(query, convert_to_numpy=True, normalize_embeddings=True)
scores = car_embeddings @ query_embedding

result_columns = [column for column in ["name", "Price", "Location", "description"] if column in cars.columns]
results = cars[result_columns].copy()
results["similarity"] = scores
results = results.sort_values("similarity", ascending=False).head(3)

print("Search query:", query)
print("Query vector shape:", query_embedding.shape)
display(results.reset_index(drop=True))

Search query: Tôi muốn tìm xe SUV 7 chỗ rộng rãi cho gia đình đi du lịch
Query vector shape: (384,)


,name,Price,Location,description,similarity
0,Honda CRV 2.0 AT - 2007,269 Triệu,Hải Phòng,"*Xe nhập khẩu, màu bạc, máy xăng 2.0 L, số tự ...",0.564087
1,Honda CRV E - 2019,699 Triệu,Hà Nội,"*Xe nhập khẩu, màu trắng, máy xăng 1.5 L, số t...",0.562782
2,Honda CRV L - 2019,785 Triệu,Hà Nội,"*Xe nhập khẩu, màu trắng, máy xăng 1.5 L, số t...",0.560511


## 6. Search again with a different query

Change `query` and rerun this cell. The car listings do not need to be encoded again.

In [18]:
def search_cars(query, top_k=3):
    if not query.strip():
        raise ValueError("The search query cannot be empty.")

    query_vector = model.encode(query.strip(), convert_to_numpy=True, normalize_embeddings=True)
    similarity = car_embeddings @ query_vector
    top_k = min(max(int(top_k), 1), len(cars))
    ranking = np.argsort(similarity)[::-1][:top_k]

    results = cars.iloc[ranking][result_columns].copy()
    results["Cosine Similarity"] = np.round(similarity[ranking], 4)
    return results.reset_index(drop=True)

query = "Tôi cần xe nhỏ gọn tiết kiệm xăng để đi làm"
display(search_cars(query))

,name,Price,Location,description,Cosine Similarity
0,Mini One 1.5 AT - 2020,1 Tỷ 200 Triệu,Hà Nội,"*Xe nhập khẩu, màu xanh, máy xăng 1.5 L, số tự...",0.5613
1,Gaz Sobol - 2024,500 Triệu,Hà Nội,"*Xe lắp ráp trong nước, màu trắng, máy dầu , s...",0.5564
2,Honda CRV G - 2024,999 Triệu,Hà Nội,"*Xe lắp ráp trong nước, màu trắng, máy xăng 1....",0.5468


## 7. Conclusion

In this assignment, a semantic search engine for car listings was built with the Sentence Transformer model `paraphrase-multilingual-MiniLM-L12-v2`.

**What was achieved:**
- The dataset of **15,442 car listings** was downloaded automatically from KaggleHub and cleaned (missing names and descriptions were handled before indexing).
- Each listing was encoded into a **384-dimensional vector**, so the whole catalogue is searchable in one matrix multiplication.
- Retrieval is performed with **cosine similarity** on normalised vectors, so the ranking reflects the *meaning* of the Vietnamese query rather than exact keyword matching.
- A reusable `search_cars(query, top_k)` function allows new queries to be run instantly without re-encoding the catalogue.

**Limitations & future work:**
- The score is a similarity measure, not a probability, and does not filter by price, location, or year.
- The quality of the results depends directly on the text fields available in the dataset; richer descriptions would give stronger embeddings.
- Next steps could include hybrid retrieval (keyword + semantic), re-ranking with a cross-encoder, and adding filters on price/location.



### Demo of the final result



In [ ]:
# Quick final demo: semantic search for a city car within budget
demo_query = "Xe ô tô 4 chỗ nhỏ gọn, giá rẻ, tiết kiệm nhiên liệu cho thành phố"
display(search_cars(demo_query, top_k=3))